# 第39章 面积图（fill_between / stackplot）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 8 / 12 步：观察关系、分布与构成**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 箱线图（boxplot）  →  **本章任务：** 面积图（fill_between / stackplot）  →  **下一步：** 饼图与环形图（pie）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时，我们常要判断“一段时间里某个数量整体是涨是跌”。


## 本章目标

学完本章，你将能够：

- **理解**：理解「面积图（fill_between / stackplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「面积图（fill_between / stackplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「面积图（fill_between / stackplot）」并读出其中的结论。


## 适用场景

**背景引入**：做数据分析时，我们常要判断“一段时间里某个数量整体是涨是跌”。折线能画出趋势，但面积图会在线下方涂上颜色，把单调的趋势变成一眼可读的“量感”，让读者更快地感受到变化的幅度与规模。本小节将用真实销售数据，带你认识面积图的两件核心工具——`fill_between`（给折线下方填充颜色）与 `stackplot`（把多个组成部分叠起来看总结构成）。 打个比方：折线图只看水面高度，面积图则把水面以下涂满——就像往杯子里倒水，多高只是线，涂满的颜色才让你看清「到底装了多少」；`stackplot` 更像往同一只杯子里分层倒不同颜色的液体，每层的厚度就是各部分占了多少。


强调趋势的累计量、区间或随时间变化的组成。


## 数据结构

有序X轴和一条或多条非负序列；堆积面积图各序列单位相同。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 alpha 参数从 0.18 改为 0.5，观察透明度对填充区域可见性的影响
2. 修改 stackplot 中的 alpha 为 0.95，对比不透明堆积与透明堆积的视觉效果
3. 在 fill_between 中添加 where 参数（如 where=(sales > 150)），观察条件填充效果


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.plot()`、`ax.fill_between()`、`ax.set()` | 强调趋势的累计量、区间或随时间变化的组成。 | 多层面积图难以比较中间序列 |
| 进阶变体 | `np.array()`、`plt.subplots()`、`ax.stackplot()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 存在负值仍直接堆积 |
| 关键参数 | `alpha` | 透明度 | 多层面积图难以比较中间序列 |
| 关键参数 | `baseline` | 堆积基线 | 存在负值仍直接堆积 |
| 关键参数 | `labels` | 组成名称 | 面积填充遮挡重要网格和文字 |
| 关键参数 | `where` | 条件填充 | 多层面积图难以比较中间序列 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-39 -->
### 数学推导｜面积堆叠必须满足组成恒等式

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜同一时点先统一粒度。** 各组数值为 $x_{1,t},\ldots,x_{G,t}$。

**第 2 步｜逐层累加形成边界。** 第 $g$ 层上边界为

$$
H_{g,t}=\sum_{j=1}^{g}x_{j,t}
$$

**第 3 步｜最上层必须回到总体。** $H_{G,t}=T_t$；若画百分比堆叠，则每层厚度为 $s_{g,t}=x_{g,t}/T_t$ 且总和为 1。

**把上面的关系收束为本章计算式：**

$$
T_t=\sum_{g=1}^{G}x_{g,t},\qquad s_{g,t}=\frac{x_{g,t}}{T_t}
$$

**符号解释：** $x_{g,t}$ 是组 $g$ 在时点 $t$ 的值，$T_t$ 是同一时点总体。

**代码对应：** 绘图前按时间透视为宽表，并检查各层之和是否等于总体。

**使用边界：** 堆叠顺序影响可读性；除最底层外，其他类别不适合比较细小变化。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, color="#1a73e8", linewidth=2)
ax.fill_between(months, sales, color="#1a73e8", alpha=0.18)
ax.set(title="上半年销售额面积图", ylabel="销售额（万元）")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：刚才的基础面积图画的是“销售额”字段 `sales`。现在请你只改一个数据字段——把面积图改成“**订单数**”字段 `orders`，并把填充透明度 `alpha` 由 `0.18` 改为 `0.5`：

1. 用 `plt.subplots` 创建新的 `fig, ax`；
2. 用 `ax.plot` + `ax.fill_between` 分别画出 `orders` 的折线和面积填充，`alpha=0.5`；
3. 用 `ax.set` 补上标题与 ylabel；
4. 最后运行自检，确认 `orders` 用的是数值字段、且与月份等长。

> 提示：`sales` 对应“销售额”（万元），`orders` 对应“订单数”（笔），两者来自上一节的按月聚合结果 `monthly_summary`，可直接使用。


In [ ]:
try:
    pass
    # 请在下方填写代码
    # 把面积图的数据字段由 sales 改成 orders，并把 alpha 改为 0.5。
    # 步骤：新建 fig/ax → ax.plot + ax.fill_between 画 orders → ax.set 补标题。

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

# 品类构成：按销售额比例拆出办公 / 数码 / 家居三条月度序列（单位：万元）
office = (sales * 0.35).round(1)
digital = (sales * 0.45).round(1)
home = (sales - office - digital).round(1)
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.stackplot(
    months,
    office,
    digital,
    home,
    labels=["办公", "数码", "家居"],
    colors=["#8ab4f8", "#81c995", "#fdd663"],
    alpha=0.85,
)
ax.set(title="销售额品类构成变化", ylabel="销售额（万元）")
ax.legend(loc="upper left", frameon=False, ncol=3)
fig.tight_layout()
plt.show()


## 参数说明

- alpha：透明度
- baseline：堆积基线
- labels：组成名称
- where：条件填充


## 结果解读

普通面积图读取边界趋势；堆积面积图读取总高度和各层厚度。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 多层面积图难以比较中间序列
- 存在负值仍直接堆积
- 面积填充遮挡重要网格和文字


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把「销售额」面积换成「利润」面积，观察波动
    # 【目标】换一个指标，练习用填充区的波动幅度来比较两个量的变化。
    import matplotlib.pyplot as plt

    # 起点示例(已可运行)：y 换成 profit，颜色换绿色系。
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(months, profit, color="#188038", linewidth=2)
    ax.fill_between(months, profit, color="#188038", alpha=0.18)
    ax.set(title="上半年利润面积图", ylabel="利润（万元）")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：换成利润后，面积起伏如何变化 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

使用填充区域表达连续趋势、区间范围或多个组成部分的累计变化。


### 你已经掌握

- 判断面积图（fill_between / stackplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `alpha` | 透明度 |
| `baseline` | 堆积基线 |
| `labels` | 组成名称 |
| `where` | 条件填充 |


### 需要注意

- 多层面积图难以比较中间序列
- 存在负值仍直接堆积
- 面积填充遮挡重要网格和文字


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
chosen_data = orders  # 切换数据字段：销售额 → 订单数
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, chosen_data, color="#1a73e8", linewidth=2)
ax.fill_between(months, chosen_data, color="#1a73e8", alpha=0.5)
ax.set(title="订单数面积图", ylabel="订单数（笔）")
fig.tight_layout()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

lower = sales * 0.9
upper = sales * 1.1
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, color="#188038", marker="o", label="预测")
ax.fill_between(
    months, lower, upper, color="#188038", alpha=0.18, label="±10%区间"
)
ax.set(title="销售预测及区间", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
